# Design Patterns

Proven, reusable solutions to common software design problems.

**In this notebook:**
- **Creational:** Singleton, Factory, Builder
- **Structural:** Decorator (GoF), Adapter
- **Behavioural:** Observer, Strategy, Command, Template Method

> Patterns are templates for thinking — not code to copy verbatim.

## Creational Patterns

### 1. Singleton

Guarantees a class has exactly one instance. Use for shared resources: config, logger, connection pool.

In [ ]:
class AppConfig:
    _instance = None

    def __new__(cls):
        if cls._instance is None:
            cls._instance = super().__new__(cls)
            cls._instance._settings = {'debug': False, 'version': '1.0'}
        return cls._instance

    def get(self, key):        return self._settings[key]
    def set(self, key, value): self._settings[key] = value

a = AppConfig()
b = AppConfig()
print(f'Same instance: {a is b}')   # True

a.set('debug', True)
print(f'b.debug: {b.get("debug")}')   # True — shared state

### 2. Factory

Creates objects without exposing instantiation logic. The caller says *what* it wants, not *how* to build it.

In [ ]:
class Dog:
    def speak(self): return 'Woof!'

class Cat:
    def speak(self): return 'Meow!'

class Bird:
    def speak(self): return 'Tweet!'

_REGISTRY = {'dog': Dog, 'cat': Cat, 'bird': Bird}

def animal_factory(kind):
    cls = _REGISTRY.get(kind.lower())
    if cls is None:
        raise ValueError(f'Unknown animal: {kind!r}')
    return cls()

for kind in ['dog', 'cat', 'bird']:
    animal = animal_factory(kind)
    print(f'{kind}: {animal.speak()}')

### 3. Builder

Separates construction of a complex object from its representation. Method chaining creates a fluent API.

In [ ]:
class QueryBuilder:
    def __init__(self):
        self._fields = ['*']
        self._table  = None
        self._where  = []
        self._limit  = None

    def select(self, *fields):
        self._fields = list(fields)
        return self   # return self enables chaining

    def from_table(self, table):
        self._table = table
        return self

    def where(self, condition):
        self._where.append(condition)
        return self

    def limit(self, n):
        self._limit = n
        return self

    def build(self):
        sql = f"SELECT {', '.join(self._fields)} FROM {self._table}"
        if self._where: sql += ' WHERE ' + ' AND '.join(self._where)
        if self._limit: sql += f' LIMIT {self._limit}'
        return sql

query = (QueryBuilder()
    .select('name', 'email')
    .from_table('users')
    .where('age > 18')
    .where('active = true')
    .limit(10)
    .build())
print(query)

## Structural Patterns

### 4. Decorator (GoF)

Wraps an object to add behaviour at runtime — distinct from Python's `@decorator` syntax, though conceptually the same.

In [ ]:
class PlainText:
    def render(self, text): return text

class TrimDecorator:
    def __init__(self, component): self._c = component
    def render(self, text): return self._c.render(text).strip()

class UpperDecorator:
    def __init__(self, component): self._c = component
    def render(self, text): return self._c.render(text).upper()

class BoldDecorator:
    def __init__(self, component): self._c = component
    def render(self, text): return f'**{self._c.render(text)}**'

# Stack decorators in any order
renderer = BoldDecorator(UpperDecorator(TrimDecorator(PlainText())))
print(renderer.render('  hello world  '))   # **HELLO WORLD**

# Different combination — different result
renderer2 = UpperDecorator(BoldDecorator(TrimDecorator(PlainText())))
print(renderer2.render('  hello world  '))  # **HELLO WORLD** (same here)

### 5. Adapter

Converts one interface into another that clients expect — like a travel plug adapter.

In [ ]:
class EuropeanSocket:
    def voltage(self):   return 220
    def plug_type(self): return 'Type C'

class USDevice:
    def run(self, socket):
        if socket.voltage() > 150:
            raise RuntimeError('Too much voltage!')
        return f'Running on {socket.voltage()}V ({socket.plug_type()})'

class Adapter:
    def __init__(self, eu_socket):
        self._s = eu_socket
    def voltage(self):   return 110       # step down
    def plug_type(self): return 'Type A'

device  = USDevice()
eu_socket = EuropeanSocket()
adapter   = Adapter(eu_socket)

print(device.run(adapter))   # Running on 110V (Type A)

## Behavioural Patterns

### 6. Observer

Objects subscribe to events from a subject and are notified when state changes.

In [ ]:
class EventEmitter:
    def __init__(self):
        self._listeners = {}

    def on(self, event, callback):
        self._listeners.setdefault(event, []).append(callback)

    def emit(self, event, **data):
        for cb in self._listeners.get(event, []):
            cb(**data)

emitter = EventEmitter()
emitter.on('sale', lambda amount, product:
    print(f'  [LOG]  Sale: {product} for R${amount:.2f}'))
emitter.on('sale', lambda amount, product:
    print(f'  [ALERT] Revenue +R${amount:.2f}') if amount > 1000 else None)

emitter.emit('sale', product='Mouse',    amount=89.90)
emitter.emit('sale', product='Notebook', amount=2500.00)

### 7. Strategy

Defines a family of algorithms, encapsulates each one, and makes them interchangeable at runtime.

In [ ]:
class ShoppingCart:
    def __init__(self):
        self._items = []

    def add(self, name, price, qty=1):
        self._items.append((name, price, qty))

    def subtotal(self):
        return sum(p * q for _, p, q in self._items)

    def checkout(self, strategy):
        sub   = self.subtotal()
        total = strategy(sub)
        print(f'  Subtotal: R${sub:.2f}  →  Total: R${total:.2f}  '
              f'(Strategy: {strategy.__name__})')
        return total

def no_discount(total):          return total
def ten_percent(total):          return total * 0.9
def bulk(total):                 return total * 0.85 if total >= 2000 else total

cart = ShoppingCart()
cart.add('Notebook', 2500, 1)
cart.add('Mouse',      90, 2)

cart.checkout(no_discount)
cart.checkout(ten_percent)
cart.checkout(bulk)

### 8. Command

Encapsulates a request as an object, enabling undo/redo, queuing, and logging.

In [ ]:
class TextEditor:
    def __init__(self):
        self.content = ''
        self._undo_stack = []

    def execute(self, command):
        command.execute(self)
        self._undo_stack.append(command)

    def undo(self):
        if self._undo_stack:
            self._undo_stack.pop().undo(self)

class AppendCommand:
    def __init__(self, text):    self._text = text
    def execute(self, editor):   editor.content += self._text
    def undo(self, editor):      editor.content = editor.content[:-len(self._text)]

class ReplaceCommand:
    def __init__(self, old, new): self._old, self._new = old, new
    def execute(self, editor):   editor.content = editor.content.replace(self._old, self._new, 1)
    def undo(self, editor):      editor.content = editor.content.replace(self._new, self._old, 1)

editor = TextEditor()
editor.execute(AppendCommand('Hello'))
editor.execute(AppendCommand(', World'))
editor.execute(ReplaceCommand('World', 'Python'))
print(editor.content)   # Hello, Python
editor.undo()
print(editor.content)   # Hello, World
editor.undo()
print(editor.content)   # Hello

### 9. Template Method

Defines the skeleton of an algorithm in a base class. Subclasses fill in specific steps without changing the overall structure.

In [ ]:
class DataProcessor:
    def process(self, data):              # template method
        raw     = self.load(data)
        cleaned = self.clean(raw)
        result  = self.transform(cleaned)
        self.save(result)
        return result

    def load(self, data):    return data
    def clean(self, data):   return data.strip()
    def transform(self, data): raise NotImplementedError
    def save(self, result):  print(f'  Saved: {result!r}')

class UpperProcessor(DataProcessor):
    def transform(self, data): return data.upper()

class ReverseProcessor(DataProcessor):
    def transform(self, data): return data[::-1]

UpperProcessor().process('  hello world  ')
ReverseProcessor().process('  hello world  ')

## Pattern Selection Quick Reference

| You want to... | Pattern |
|---|---|
| One shared instance (config, logger) | Singleton |
| Create objects without knowing the class | Factory |
| Build complex objects step by step | Builder |
| Add behaviour without subclassing | Decorator (GoF) |
| Make incompatible interfaces work | Adapter |
| Notify many objects on state change | Observer |
| Swap algorithms at runtime | Strategy |
| Support undo/redo, action queuing | Command |
| Common algorithm, variable steps | Template Method |

## Practice

| File | Difficulty | Topics |
|---|---|---|
| [01-easy.py](exercises/01-easy.py) | Easy | Singleton, Factory, Template Method |
| [02-medium.py](exercises/02-medium.py) | Medium | Observer, Strategy, GoF Decorator |
| [03-challenge.py](exercises/03-challenge.py) | Challenge | Builder (SQL), Command (undo/redo), Plugin Registry |

Solutions: [solutions/](solutions/)